# Best Architecture Model: C201

In [32]:
# ============================================================
# Imports
# ============================================================
import os
import csv
import time
import random
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, _LRScheduler
from torch.cuda.amp import autocast, GradScaler

from PIL import Image, ImageEnhance, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] device = {device}, gpu count = {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")


[INFO] device = cuda, gpu count = 2
[INFO] GPU: Tesla T4


In [49]:
# ============================================================
# Config  
# Default matches the README's best-CIFAR-10 command:
#   python main.py --dataset c10 --label-smoothing --autoaugment
# ============================================================
QUICK_TEST = False  # set True first to sanity-check the whole pipeline in a few minutes

class Args:
    # data / model
    dataset        = "c10"        # "c10", "c100", "svhn"
    model_name     = "vit"
    patch          = 8
    batch_size     = 128
    eval_batch_size = 1024
    dropout        = 0.0
    head           = 4
    num_layers     = 2
    hidden         = 96
    mlp_hidden     = 96
    off_cls_token  = False        # True -> use mean pooling instead of CLS token

    # optimization
    lr             = 1e-3
    min_lr         = 1e-5
    beta1          = 0.9
    beta2          = 0.999
    weight_decay   = 5e-5
    warmup_epoch   = 5
    max_epochs     = 120
    precision      = 16           # 16 -> mixed precision (if CUDA available)

    # loss
    criterion      = "ce"
    label_smoothing = True
    smoothing      = 0.1

    # augmentation
    autoaugment    = True
    rcpaste        = False
    cutmix         = False
    mixup          = False

    # misc
    off_benchmark  = False
    dry_run        = False
    seed           = 42
    api_key        = None         # comet.ml disabled -> CSV logging is used instead
    project_name   = "VisionTransformer"

args = Args()

if QUICK_TEST:
    args.max_epochs = 3
    args.warmup_epoch = 1
    print("[INFO] QUICK_TEST=True -> max_epochs reduced to 3 for a fast pipeline check.")

# derived fields (same logic as main.py)
args.benchmark = not args.off_benchmark
args.gpus = torch.cuda.device_count()
_cpu_count = os.cpu_count() or 4
args.num_workers = min(4 * args.gpus, _cpu_count) if args.gpus else min(8, _cpu_count)
args.is_cls_token = not args.off_cls_token
if not args.gpus:
    args.precision = 32

if args.mlp_hidden != args.hidden * 4:
    print(f"[INFO] In the original paper, mlp_hidden(CURRENT:{args.mlp_hidden}) is usually hidden*4={args.hidden*4}. "
          f"This repo intentionally deviates from that (default mlp_hidden=384).")

torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)
torch.backends.cudnn.benchmark = args.benchmark

RESUME = False  # if True, auto-resumes from a saved "last" checkpoint when present
LOG_DIR = "/kaggle/working/logs"
CKPT_DIR = "/kaggle/working/weights"
DATA_ROOT = "/kaggle/working/data"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)


[INFO] In the original paper, mlp_hidden(CURRENT:96) is usually hidden*4=384. This repo intentionally deviates from that (default mlp_hidden=384).


In [34]:
# ============================================================
# ops.py  (AutoAugment primitive ops)
# Fix applied: np.int -> int (np.int was removed in numpy>=1.24)
# ============================================================
class ShearX(object):
    def __init__(self, fillcolor=(128, 128, 128)):
        self.fillcolor = fillcolor

    def __call__(self, x, magnitude):
        return x.transform(
            x.size, Image.AFFINE, (1, magnitude * random.choice([-1, 1]), 0, 0, 1, 0),
            Image.BICUBIC, fillcolor=self.fillcolor)


class ShearY(object):
    def __init__(self, fillcolor=(128, 128, 128)):
        self.fillcolor = fillcolor

    def __call__(self, x, magnitude):
        return x.transform(
            x.size, Image.AFFINE, (1, 0, 0, magnitude * random.choice([-1, 1]), 1, 0),
            Image.BICUBIC, fillcolor=self.fillcolor)


class TranslateX(object):
    def __init__(self, fillcolor=(128, 128, 128)):
        self.fillcolor = fillcolor

    def __call__(self, x, magnitude):
        return x.transform(
            x.size, Image.AFFINE, (1, 0, magnitude * x.size[0] * random.choice([-1, 1]), 0, 1, 0),
            fillcolor=self.fillcolor)


class TranslateY(object):
    def __init__(self, fillcolor=(128, 128, 128)):
        self.fillcolor = fillcolor

    def __call__(self, x, magnitude):
        return x.transform(
            x.size, Image.AFFINE, (1, 0, 0, 0, 1, magnitude * x.size[1] * random.choice([-1, 1])),
            fillcolor=self.fillcolor)


class Rotate(object):
    def __call__(self, x, magnitude):
        rot = x.convert("RGBA").rotate(magnitude)
        return Image.composite(rot, Image.new("RGBA", rot.size, (128,) * 4), rot).convert(x.mode)


class Color(object):
    def __call__(self, x, magnitude):
        return ImageEnhance.Color(x).enhance(1 + magnitude * random.choice([-1, 1]))


class Posterize(object):
    def __call__(self, x, magnitude):
        return ImageOps.posterize(x, magnitude)


class Solarize(object):
    def __call__(self, x, magnitude):
        return ImageOps.solarize(x, magnitude)


class Contrast(object):
    def __call__(self, x, magnitude):
        return ImageEnhance.Contrast(x).enhance(1 + magnitude * random.choice([-1, 1]))


class Sharpness(object):
    def __call__(self, x, magnitude):
        return ImageEnhance.Sharpness(x).enhance(1 + magnitude * random.choice([-1, 1]))


class Brightness(object):
    def __call__(self, x, magnitude):
        return ImageEnhance.Brightness(x).enhance(1 + magnitude * random.choice([-1, 1]))


class AutoContrast(object):
    def __call__(self, x, magnitude):
        return ImageOps.autocontrast(x)


class Equalize(object):
    def __call__(self, x, magnitude):
        return ImageOps.equalize(x)


class Invert(object):
    def __call__(self, x, magnitude):
        return ImageOps.invert(x)


In [35]:
# ============================================================
# autoaugment.py  (CIFAR10Policy is what we use; kept ImageNet/SVHN too)
# ============================================================
class SubPolicy(object):
    def __init__(self, p1, operation1, magnitude_idx1, p2, operation2, magnitude_idx2, fillcolor=(128, 128, 128)):
        ranges = {
            "shearX": np.linspace(0, 0.3, 10),
            "shearY": np.linspace(0, 0.3, 10),
            "translateX": np.linspace(0, 150 / 331, 10),
            "translateY": np.linspace(0, 150 / 331, 10),
            "rotate": np.linspace(0, 30, 10),
            "color": np.linspace(0.0, 0.9, 10),
            "posterize": np.round(np.linspace(8, 4, 10), 0).astype(int),   # fixed: np.int -> int
            "solarize": np.linspace(256, 0, 10),
            "contrast": np.linspace(0.0, 0.9, 10),
            "sharpness": np.linspace(0.0, 0.9, 10),
            "brightness": np.linspace(0.0, 0.9, 10),
            "autocontrast": [0] * 10,
            "equalize": [0] * 10,
            "invert": [0] * 10
        }

        func = {
            "shearX": ShearX(fillcolor=fillcolor),
            "shearY": ShearY(fillcolor=fillcolor),
            "translateX": TranslateX(fillcolor=fillcolor),
            "translateY": TranslateY(fillcolor=fillcolor),
            "rotate": Rotate(),
            "color": Color(),
            "posterize": Posterize(),
            "solarize": Solarize(),
            "contrast": Contrast(),
            "sharpness": Sharpness(),
            "brightness": Brightness(),
            "autocontrast": AutoContrast(),
            "equalize": Equalize(),
            "invert": Invert()
        }

        self.p1 = p1
        self.operation1 = func[operation1]
        self.magnitude1 = ranges[operation1][magnitude_idx1]
        self.p2 = p2
        self.operation2 = func[operation2]
        self.magnitude2 = ranges[operation2][magnitude_idx2]

    def __call__(self, img):
        if random.random() < self.p1:
            img = self.operation1(img, self.magnitude1)
        if random.random() < self.p2:
            img = self.operation2(img, self.magnitude2)
        return img


class CIFAR10Policy(object):
    """Randomly choose one of the best 25 Sub-policies on CIFAR10."""
    def __init__(self, fillcolor=(128, 128, 128)):
        self.policies = [
            SubPolicy(0.1, "invert", 7, 0.2, "contrast", 6, fillcolor),
            SubPolicy(0.7, "rotate", 2, 0.3, "translateX", 9, fillcolor),
            SubPolicy(0.8, "sharpness", 1, 0.9, "sharpness", 3, fillcolor),
            SubPolicy(0.5, "shearY", 8, 0.7, "translateY", 9, fillcolor),
            SubPolicy(0.5, "autocontrast", 8, 0.9, "equalize", 2, fillcolor),

            SubPolicy(0.2, "shearY", 7, 0.3, "posterize", 7, fillcolor),
            SubPolicy(0.4, "color", 3, 0.6, "brightness", 7, fillcolor),
            SubPolicy(0.3, "sharpness", 9, 0.7, "brightness", 9, fillcolor),
            SubPolicy(0.6, "equalize", 5, 0.5, "equalize", 1, fillcolor),
            SubPolicy(0.6, "contrast", 7, 0.6, "sharpness", 5, fillcolor),

            SubPolicy(0.7, "color", 7, 0.5, "translateX", 8, fillcolor),
            SubPolicy(0.3, "equalize", 7, 0.4, "autocontrast", 8, fillcolor),
            SubPolicy(0.4, "translateY", 3, 0.2, "sharpness", 6, fillcolor),
            SubPolicy(0.9, "brightness", 6, 0.2, "color", 8, fillcolor),
            SubPolicy(0.5, "solarize", 2, 0.0, "invert", 3, fillcolor),

            SubPolicy(0.2, "equalize", 0, 0.6, "autocontrast", 0, fillcolor),
            SubPolicy(0.2, "equalize", 8, 0.6, "equalize", 4, fillcolor),
            SubPolicy(0.9, "color", 9, 0.6, "equalize", 6, fillcolor),
            SubPolicy(0.8, "autocontrast", 4, 0.2, "solarize", 8, fillcolor),
            SubPolicy(0.1, "brightness", 3, 0.7, "color", 0, fillcolor),

            SubPolicy(0.4, "solarize", 5, 0.9, "autocontrast", 3, fillcolor),
            SubPolicy(0.9, "translateY", 9, 0.7, "translateY", 9, fillcolor),
            SubPolicy(0.9, "autocontrast", 2, 0.8, "solarize", 3, fillcolor),
            SubPolicy(0.8, "equalize", 8, 0.1, "invert", 3, fillcolor),
            SubPolicy(0.7, "translateY", 9, 0.9, "autocontrast", 1, fillcolor)
        ]

    def __call__(self, img):
        policy_idx = random.randint(0, len(self.policies) - 1)
        return self.policies[policy_idx](img)

    def __repr__(self):
        return "AutoAugment CIFAR10 Policy"


In [36]:
# ============================================================
# da.py  (RandomCropPaste, CutMix, MixUp)
# Fix applied: np.int -> int in RandomCropPaste._rand_bbox
# ============================================================
class RandomCropPaste(object):
    def __init__(self, size, alpha=1.0, flip_p=0.5):
        """Randomly flip and paste a cropped image on the same image."""
        self.size = size
        self.alpha = alpha
        self.flip_p = flip_p

    def __call__(self, img):
        lam = np.random.beta(self.alpha, self.alpha)
        front_bbx1, front_bby1, front_bbx2, front_bby2 = self._rand_bbox(lam)
        img_front = img[:, front_bby1:front_bby2, front_bbx1:front_bbx2].clone()
        front_w = front_bbx2 - front_bbx1
        front_h = front_bby2 - front_bby1

        img_x1 = np.random.randint(0, high=max(self.size - front_w, 1))
        img_y1 = np.random.randint(0, high=max(self.size - front_h, 1))
        img_x2 = img_x1 + front_w
        img_y2 = img_y1 + front_h

        if np.random.rand(1) <= self.flip_p:
            img_front = img_front.flip((-1,))
        if np.random.rand(1) <= self.flip_p:
            img = img.flip((-1,))

        mixup_alpha = np.random.rand(1)
        img[:, img_y1:img_y2, img_x1:img_x2] *= mixup_alpha
        img[:, img_y1:img_y2, img_x1:img_x2] += img_front * (1 - mixup_alpha)
        return img

    def _rand_bbox(self, lam):
        W = self.size
        H = self.size
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)     # fixed: np.int -> int
        cut_h = int(H * cut_rat)     # fixed: np.int -> int

        cx = np.random.randint(W)
        cy = np.random.randint(H)

        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        return bbx1, bby1, bbx2, bby2


class CutMix(object):
    def __init__(self, size, beta):
        self.size = size
        self.beta = beta

    def __call__(self, batch):
        img, label = batch
        rand_img, rand_label = self._shuffle_minibatch(batch)
        lambda_ = np.random.beta(self.beta, self.beta)
        r_x = np.random.uniform(0, self.size)
        r_y = np.random.uniform(0, self.size)
        r_w = self.size * np.sqrt(1 - lambda_)
        r_h = self.size * np.sqrt(1 - lambda_)
        x1 = int(np.clip(r_x - r_w // 2, a_min=0, a_max=self.size))
        x2 = int(np.clip(r_x + r_w // 2, a_min=0, a_max=self.size))
        y1 = int(np.clip(r_y - r_h // 2, a_min=0, a_max=self.size))
        y2 = int(np.clip(r_y + r_h // 2, a_min=0, a_max=self.size))
        img[:, :, x1:x2, y1:y2] = rand_img[:, :, x1:x2, y1:y2]

        lambda_ = 1 - (x2 - x1) * (y2 - y1) / (self.size * self.size)
        return img, label, rand_label, lambda_

    def _shuffle_minibatch(self, batch):
        img, label = batch
        rand_img, rand_label = img.clone(), label.clone()
        rand_idx = torch.randperm(img.size(0))
        rand_img, rand_label = rand_img[rand_idx], rand_label[rand_idx]
        return rand_img, rand_label


# Code: https://github.com/facebookresearch/mixup-cifar10
class MixUp(object):
    def __init__(self, alpha=0.1):
        self.alpha = alpha

    def __call__(self, batch):
        """Returns mixed inputs, pairs of targets, and lambda"""
        x, y = batch
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = x.size(0)
        index = torch.randperm(batch_size)
        mixed_x = lam * x + (1 - lam) * x[index, :]
        y_a, y_b = y, y[index]
        return mixed_x, y_a, y_b, lam


In [37]:
# ============================================================
# criterions.py
# ============================================================
class LabelSmoothingCrossEntropyLoss(nn.Module):
    def __init__(self, classes, smoothing=0.0, dim=-1):
        super(LabelSmoothingCrossEntropyLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))


In [38]:
# ============================================================
# layers.py  (Transformer encoder + multi-head self-attention)
# ============================================================
class TransformerEncoder(nn.Module):
    def __init__(self, feats: int, mlp_hidden: int, head: int = 8, dropout: float = 0.):
        super(TransformerEncoder, self).__init__()
        self.la1 = nn.LayerNorm(feats)
        self.msa = MultiHeadSelfAttention(feats, head=head, dropout=dropout)
        self.la2 = nn.LayerNorm(feats)
        self.mlp = nn.Sequential(
            nn.Linear(feats, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, feats),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        out = self.msa(self.la1(x)) + x
        out = self.mlp(self.la2(out)) + out
        return out


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, feats: int, head: int = 8, dropout: float = 0.):
        super(MultiHeadSelfAttention, self).__init__()
        self.head = head
        self.feats = feats
        self.sqrt_d = self.feats ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)

        self.o = nn.Linear(feats, feats)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, n, f = x.size()
        q = self.q(x).view(b, n, self.head, self.feats // self.head).transpose(1, 2)
        k = self.k(x).view(b, n, self.head, self.feats // self.head).transpose(1, 2)
        v = self.v(x).view(b, n, self.head, self.feats // self.head).transpose(1, 2)

        score = F.softmax(torch.einsum("bhif, bhjf->bhij", q, k) / self.sqrt_d, dim=-1)  # (b,h,n,n)
        attn = torch.einsum("bhij, bhjf->bihf", score, v)  # (b,n,h,f//h)
        o = self.dropout(self.o(attn.flatten(2)))
        return o


In [39]:
# ============================================================
# vit.py  (the ViT model itself)
# ============================================================
class ViT(nn.Module):
    def __init__(self, in_c: int = 3, num_classes: int = 10, img_size: int = 32, patch: int = 8,
                 dropout: float = 0., num_layers: int = 7, hidden: int = 384, mlp_hidden: int = 384 * 4,
                 head: int = 8, is_cls_token: bool = True):
        super(ViT, self).__init__()

        self.patch = patch  # number of patches in one row (or col)
        self.is_cls_token = is_cls_token
        self.patch_size = img_size // self.patch
        f = (img_size // self.patch) ** 2 * 3  # patch vec length
        num_tokens = (self.patch ** 2) + 1 if self.is_cls_token else (self.patch ** 2)

        self.emb = nn.Linear(f, hidden)  # (b, n, f)
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden)) if is_cls_token else None
        self.pos_emb = nn.Parameter(torch.randn(1, num_tokens, hidden))
        enc_list = [TransformerEncoder(hidden, mlp_hidden=mlp_hidden, dropout=dropout, head=head) for _ in range(num_layers)]
        self.enc = nn.Sequential(*enc_list)
        self.fc = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x):
        out = self._to_words(x)
        out = self.emb(out)
        if self.is_cls_token:
            out = torch.cat([self.cls_token.repeat(out.size(0), 1, 1), out], dim=1)
        out = out + self.pos_emb
        out = self.enc(out)
        if self.is_cls_token:
            out = out[:, 0]
        else:
            out = out.mean(1)
        out = self.fc(out)
        return out

    def _to_words(self, x):
        """(b, c, h, w) -> (b, n, f)"""
        out = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size).permute(0, 2, 3, 4, 5, 1)
        out = out.reshape(x.size(0), self.patch ** 2, -1)
        return out


In [40]:
# ============================================================
# utils.py  (get_transform / get_dataset / get_model / get_criterion / get_experiment_name)
# ============================================================
def get_criterion(args):
    if args.criterion == "ce":
        if args.label_smoothing:
            criterion = LabelSmoothingCrossEntropyLoss(args.num_classes, smoothing=args.smoothing)
        else:
            criterion = nn.CrossEntropyLoss()
    else:
        raise ValueError(f"{args.criterion}?")
    return criterion


def get_model(args):
    if args.model_name == "vit":
        net = ViT(
            args.in_c,
            args.num_classes,
            img_size=args.size,
            patch=args.patch,
            dropout=args.dropout,
            mlp_hidden=args.mlp_hidden,
            num_layers=args.num_layers,
            hidden=args.hidden,
            head=args.head,
            is_cls_token=args.is_cls_token
        )
    else:
        raise NotImplementedError(f"{args.model_name} is not implemented yet...")
    return net


def get_transform(args):
    train_transform = []
    test_transform = []
    train_transform += [transforms.RandomCrop(size=args.size, padding=args.padding)]
    if args.dataset != "svhn":
        train_transform += [transforms.RandomHorizontalFlip()]

    if args.autoaugment:
        if args.dataset in ("c10", "c100"):
            train_transform.append(CIFAR10Policy())
        else:
            print(f"No AutoAugment for {args.dataset}")

    train_transform += [
        transforms.ToTensor(),
        transforms.Normalize(mean=args.mean, std=args.std)
    ]
    if args.rcpaste:
        train_transform += [RandomCropPaste(size=args.size)]

    test_transform += [
        transforms.ToTensor(),
        transforms.Normalize(mean=args.mean, std=args.std)
    ]

    train_transform = transforms.Compose(train_transform)
    test_transform = transforms.Compose(test_transform)
    return train_transform, test_transform


def get_dataset(args):
    root = DATA_ROOT
    if args.dataset == "c10":
        args.in_c = 3
        args.num_classes = 10
        args.size = 32
        args.padding = 4
        args.mean, args.std = [0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]
        train_transform, test_transform = get_transform(args)
        train_ds = torchvision.datasets.CIFAR10(root, train=True, transform=train_transform, download=True)
        test_ds = torchvision.datasets.CIFAR10(root, train=False, transform=test_transform, download=True)
    else:
        raise NotImplementedError(f"{args.dataset} is not implemented yet.")

    return train_ds, test_ds


def get_experiment_name(args):
    experiment_name = f"{args.model_name}_{args.dataset}"
    if args.autoaugment:
        experiment_name += "_aa"
    if args.label_smoothing:
        experiment_name += "_ls"
    if args.rcpaste:
        experiment_name += "_rc"
    if args.cutmix:
        experiment_name += "_cm"
    if args.mixup:
        experiment_name += "_mu"
    if args.off_cls_token:
        experiment_name += "_gap"
    return experiment_name


In [41]:
# ============================================================
# Warmup + Cosine LR schedule
# Re-implementation of the linear-warmup-then-handoff behaviour that the
# original repo got from the `warmup_scheduler` package (GradualWarmupScheduler,
# multiplier=1.0), so nothing extra needs to be installed.
# ============================================================
class GradualWarmupScheduler(_LRScheduler):
    def __init__(self, optimizer, total_epoch, after_scheduler=None):
        self.total_epoch = total_epoch
        self.after_scheduler = after_scheduler
        self.finished = False
        super().__init__(optimizer)

    def get_lr(self):
        if self.last_epoch >= self.total_epoch:
            if self.after_scheduler is not None:
                if not self.finished:
                    self.after_scheduler.base_lrs = self.base_lrs
                    self.finished = True
                return self.after_scheduler.get_last_lr()
            return self.base_lrs
        return [base_lr * (self.last_epoch / self.total_epoch) for base_lr in self.base_lrs]

    def step(self, epoch=None):
        if self.finished and self.after_scheduler is not None:
            if epoch is None:
                self.after_scheduler.step(None)
            else:
                self.after_scheduler.step(epoch - self.total_epoch)
            self._last_lr = self.after_scheduler.get_last_lr()
        else:
            super().step(epoch)


In [ ]:
# ============================================================
# Build dataset + dataloaders
# ============================================================
experiment_name = get_experiment_name(args)
print(f"Experiment: {experiment_name}")

train_ds, test_ds = get_dataset(args)
print(f"[INFO] train size={len(train_ds)}, test size={len(test_ds)}, num_classes={args.num_classes}")

train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                       num_workers=args.num_workers, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=args.eval_batch_size, shuffle=False,
                      num_workers=args.num_workers, pin_memory=True)


In [23]:
# ============================================================
# Build model, optimizer, scheduler, criterion
# ============================================================
model = get_model(args).to(device)
if args.gpus > 1:
    print(f"[INFO] Using {args.gpus} GPUs via DataParallel")
    model = nn.DataParallel(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params/1e6:.2f}M")

optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(args.beta1, args.beta2),
                              weight_decay=args.weight_decay)
base_scheduler = CosineAnnealingLR(optimizer, T_max=args.max_epochs, eta_min=args.min_lr)
scheduler = GradualWarmupScheduler(optimizer, total_epoch=args.warmup_epoch, after_scheduler=base_scheduler)
criterion = get_criterion(args)

use_amp = (args.precision == 16) and torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)
print(f"[INFO] Mixed precision (AMP) enabled: {use_amp}")

cutmix_fn = CutMix(args.size, beta=1.) if args.cutmix else None
mixup_fn = MixUp(alpha=1.) if args.mixup else None


AttributeError: 'Args' object has no attribute 'in_c'

In [ ]:
# ============================================================
# Train / eval epoch functions
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, scaler, use_amp, cutmix=None, mixup=None):
    model.train()
    running_loss, running_correct, running_n = 0.0, 0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for img, label in pbar:
        img = img.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)

        lam, label_b = 1.0, label
        if cutmix is not None:
            img, label, label_b, lam = cutmix((img, label))
        elif mixup is not None:
            if np.random.rand() <= 0.8:
                img, label, label_b, lam = mixup((img, label))
            else:
                label_b = label

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            out = model(img)
            loss = criterion(out, label) * lam + criterion(out, label_b) * (1. - lam)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = img.size(0)
        running_loss += loss.item() * bs
        running_correct += (out.argmax(-1) == label).float().sum().item()
        running_n += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / running_n, running_correct / running_n


@torch.no_grad()
def evaluate(model, loader, criterion, use_amp):
    model.eval()
    running_loss, running_correct, running_n = 0.0, 0, 0
    pbar = tqdm(loader, desc="val", leave=False)
    for img, label in pbar:
        img = img.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            out = model(img)
            loss = criterion(out, label)
        bs = img.size(0)
        running_loss += loss.item() * bs
        running_correct += (out.argmax(-1) == label).float().sum().item()
        running_n += bs
    return running_loss / running_n, running_correct / running_n


In [ ]:
# ============================================================
# Main training loop
# Saves a CSV log + "last"/"best" checkpoints every epoch, and auto-resumes
# from "last" if this cell is re-run (handy given Kaggle session time limits).
# ============================================================
log_path = os.path.join(LOG_DIR, f"{experiment_name}.csv")
ckpt_last_path = os.path.join(CKPT_DIR, f"{experiment_name}_last.pth")
ckpt_best_path = os.path.join(CKPT_DIR, f"{experiment_name}_best.pth")

start_epoch = 0
best_acc = 0.0

if RESUME and os.path.exists(ckpt_last_path):
    ckpt = torch.load(ckpt_last_path, map_location=device)
    (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]
    print(f"[INFO] Resumed from checkpoint: epoch {start_epoch}, best_acc so far {best_acc*100:.2f}%")

if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc", "epoch_time_sec"])

for epoch in range(start_epoch, args.max_epochs):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_dl, optimizer, criterion, scaler, use_amp,
                                             cutmix=cutmix_fn, mixup=mixup_fn)
    val_loss, val_acc = evaluate(model, test_dl, criterion, use_amp)
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0

    print(f"[Epoch {epoch+1}/{args.max_epochs}] lr={lr_now:.6f} "
          f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% "
          f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% ({dt:.1f}s)")

    with open(log_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, lr_now, train_loss, train_acc, val_loss, val_acc, dt])

    is_best = val_acc > best_acc
    best_acc = max(best_acc, val_acc)

    ckpt_dict = {
        "epoch": epoch,
        "model": (model.module if isinstance(model, nn.DataParallel) else model).state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_acc": best_acc,
    }
    torch.save(ckpt_dict, ckpt_last_path)
    if is_best:
        torch.save(ckpt_dict, ckpt_best_path)

print(f"[DONE] Best val accuracy: {best_acc*100:.2f}%  (README reference: ~90.92% on CIFAR-10 @ 200 epochs)")
print(f"Checkpoints: {ckpt_last_path}, {ckpt_best_path}")
print(f"Log CSV: {log_path}")


In [ ]:
# ============================================================
# Plot loss / accuracy curves from the CSV log
# ============================================================
epochs_, lrs_, tr_loss_, tr_acc_, va_loss_, va_acc_ = [], [], [], [], [], []
with open(log_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs_.append(int(row["epoch"]))
        lrs_.append(float(row["lr"]))
        tr_loss_.append(float(row["train_loss"]))
        tr_acc_.append(float(row["train_acc"]))
        va_loss_.append(float(row["val_loss"]))
        va_acc_.append(float(row["val_acc"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_, tr_loss_, label="train loss")
axes[0].plot(epochs_, va_loss_, label="val loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs_, [a * 100 for a in tr_acc_], label="train acc")
axes[1].plot(epochs_, [a * 100 for a in va_acc_], label="val acc")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy (%)"); axes[1].set_title("Accuracy"); axes[1].legend()

plt.tight_layout()
curve_path = f"/kaggle/working/{experiment_name}_curves.png"
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved curves to {curve_path}")
if va_acc_:
    print(f"Best val acc from log: {max(va_acc_)*100:.2f}%")


# C201 CKPT

In [55]:
# ============================================================
# DAY 2 — C201 FP32 CHECKPOINT LOADING
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ============================================================
# 1. EXACT MULTI-HEAD SELF-ATTENTION
# ============================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        feats,
        heads,
        dropout=0.0
    ):
        super().__init__()

        self.feats = feats
        self.heads = heads

        assert feats % heads == 0

        self.head_dim = feats // heads

        # IMPORTANT:
        # The original trained model divides attention
        # scores by sqrt(feats), NOT sqrt(head_dim).
        self.sqrt_d = feats ** 0.5

        self.q = nn.Linear(
            feats,
            feats
        )

        self.k = nn.Linear(
            feats,
            feats
        )

        self.v = nn.Linear(
            feats,
            feats
        )

        self.o = nn.Linear(
            feats,
            feats
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        B, N, C = x.shape

        # Q, K, V
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        # Split into heads
        q = q.reshape(
            B,
            N,
            self.heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.reshape(
            B,
            N,
            self.heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.reshape(
            B,
            N,
            self.heads,
            self.head_dim
        ).transpose(1, 2)

        # Attention scores
        attn = torch.matmul(
            q,
            k.transpose(-2, -1)
        )

        # EXACT scaling used during training
        attn = attn / self.sqrt_d

        # Softmax
        attn = torch.softmax(
            attn,
            dim=-1
        )

        attn = self.dropout(attn)

        # Attention output
        out = torch.matmul(
            attn,
            v
        )

        # Merge heads
        out = out.transpose(
            1,
            2
        ).reshape(
            B,
            N,
            C
        )

        # Output projection
        out = self.o(out)

        return out


# ============================================================
# 2. EXACT TRANSFORMER ENCODER
# ============================================================

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        feats,
        heads,
        mlp_hidden,
        dropout=0.0
    ):
        super().__init__()

        # IMPORTANT:
        # These names MUST match the checkpoint:
        # la1, msa, la2
        self.la1 = nn.LayerNorm(feats)

        self.msa = MultiHeadSelfAttention(
            feats,
            heads,
            dropout
        )

        self.la2 = nn.LayerNorm(feats)

        self.mlp = nn.Sequential(

            nn.Linear(
                feats,
                mlp_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                mlp_hidden,
                feats
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )


    def forward(self, x):

        # Pre-LN attention + residual
        x = x + self.msa(
            self.la1(x)
        )

        # Pre-LN MLP + residual
        x = x + self.mlp(
            self.la2(x)
        )

        return x


# ============================================================
# 3. EXACT ViT
# ============================================================

class ViT(nn.Module):

    def __init__(
        self,
        in_c=3,
        num_classes=10,
        img_size=32,
        patch=8,
        dropout=0.0,
        num_layers=7,
        hidden=384,
        mlp_hidden=384,
        head=12,
        is_cls_token=True
    ):
        super().__init__()

        self.img_size = img_size
        self.patch = patch

        self.patch_size = (
            img_size // patch
        )

        self.is_cls_token = is_cls_token

        # ----------------------------------------------------
        # Patch dimension
        # ----------------------------------------------------

        f = (
            self.patch_size
            * self.patch_size
            * in_c
        )

        # ----------------------------------------------------
        # Patch embedding
        # ----------------------------------------------------

        self.emb = nn.Linear(
            f,
            hidden
        )

        # ----------------------------------------------------
        # Number of tokens
        # ----------------------------------------------------

        num_tokens = (
            patch ** 2
            + (1 if is_cls_token else 0)
        )

        # ----------------------------------------------------
        # CLS token
        # ----------------------------------------------------

        if is_cls_token:

            self.cls_token = nn.Parameter(
                torch.zeros(
                    1,
                    1,
                    hidden
                )
            )

        else:

            self.cls_token = None

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        self.pos_emb = nn.Parameter(
            torch.zeros(
                1,
                num_tokens,
                hidden
            )
        )

        # ----------------------------------------------------
        # Transformer encoder
        # ----------------------------------------------------

        self.enc = nn.Sequential(

            *[
                TransformerEncoder(
                    hidden,
                    head,
                    mlp_hidden,
                    dropout
                )

                for _ in range(num_layers)
            ]
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.fc = nn.Sequential(

            nn.LayerNorm(
                hidden
            ),

            nn.Linear(
                hidden,
                num_classes
            )
        )


    def forward(self, img):

        B, C, H, W = img.shape

        # ----------------------------------------------------
        # Extract image patches
        # ----------------------------------------------------

        p = self.patch_size

        x = img.unfold(
            2,
            p,
            p
        ).unfold(
            3,
            p,
            p
        )

        # B, C, patch_rows, patch_cols, p, p

        x = x.permute(
            0,
            2,
            3,
            1,
            4,
            5
        )

        # B, patch_rows, patch_cols, C, p, p

        x = x.reshape(
            B,
            self.patch ** 2,
            -1
        )

        # ----------------------------------------------------
        # Patch embedding
        # ----------------------------------------------------

        x = self.emb(x)

        # ----------------------------------------------------
        # CLS token
        # ----------------------------------------------------

        if self.is_cls_token:

            cls = self.cls_token.expand(
                B,
                -1,
                -1
            )

            x = torch.cat(
                [
                    cls,
                    x
                ],
                dim=1
            )

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        x = x + self.pos_emb

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        x = self.enc(x)

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        if self.is_cls_token:

            x = x[:, 0]

        else:

            x = x.mean(
                dim=1
            )

        x = self.fc(x)

        return x


# ============================================================
# 4. LOAD TRAINED C201 CHECKPOINT
# ============================================================

CKPT = (
    "/kaggle/input/datasets/tiw775/"
    "weights/vit_c10_aa_ls_best.pth"
)

checkpoint = torch.load(
    CKPT,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint loaded")
print("Checkpoint keys:")
print(checkpoint.keys())

print(
    "Best validation accuracy:",
    checkpoint["best_acc"]
)


# ============================================================
# 5. CREATE EXACT C201
# ============================================================

model_fp32 = ViT(

    in_c=3,

    num_classes=10,

    img_size=32,

    patch=8,

    dropout=0.0,

    num_layers=2,

    hidden=96,

    mlp_hidden=96,

    head=4,

    is_cls_token=True

).to(device)


# ============================================================
# 6. LOAD WEIGHTS
# ============================================================

model_fp32.load_state_dict(
    checkpoint["model"]
)

model_fp32.eval()


# ============================================================
# 7. VERIFY
# ============================================================

total_params = sum(
    p.numel()
    for p in model_fp32.parameters()
)

print("\n========================================")
print("C201 FP32 MODEL LOADED SUCCESSFULLY")
print("========================================")

print(
    "Parameters:",
    total_params
)

print(
    "Best validation accuracy:",
    checkpoint["best_acc"]
)


# ============================================================
# 8. VERIFY FORWARD PASS
# ============================================================

dummy = torch.randn(
    1,
    3,
    32,
    32
).to(device)

with torch.no_grad():

    output = model_fp32(
        dummy
    )

print(
    "Input shape:",
    dummy.shape
)

print(
    "Output shape:",
    output.shape
)

assert total_params == 124714
assert output.shape == (1, 10)

print("\nALL CHECKS PASSED")

Device: cuda

Checkpoint loaded
Checkpoint keys:
dict_keys(['epoch', 'model', 'optimizer', 'scheduler', 'scaler', 'best_acc'])
Best validation accuracy: 0.7695

C201 FP32 MODEL LOADED SUCCESSFULLY
Parameters: 124714
Best validation accuracy: 0.7695
Input shape: torch.Size([1, 3, 32, 32])
Output shape: torch.Size([1, 10])

ALL CHECKS PASSED


In [56]:
# ============================================================
# DAY 2 — INT8 CALIBRATION
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import json
import os

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CALIBRATION_SAMPLES = 512
BATCH_SIZE = 64

# ------------------------------------------------------------
# CIFAR-10 preprocessing
# Must match model input preprocessing
# ------------------------------------------------------------

CIFAR_MEAN = (
    0.4914,
    0.4822,
    0.4465
)

CIFAR_STD = (
    0.2470,
    0.2435,
    0.2616
)

calibration_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        CIFAR_MEAN,
        CIFAR_STD
    )
])


# ------------------------------------------------------------
# Load CIFAR-10 training set
# ------------------------------------------------------------

calibration_dataset_full = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=calibration_transform
)


# ------------------------------------------------------------
# Deterministic calibration subset
# ------------------------------------------------------------

generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(calibration_dataset_full),
    generator=generator
)[:CALIBRATION_SAMPLES]

calibration_dataset = Subset(
    calibration_dataset_full,
    indices.tolist()
)


calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=" * 60)
print("INT8 CALIBRATION DATASET")
print("=" * 60)

print(
    "Total calibration samples:",
    len(calibration_dataset)
)

print(
    "Batch size:",
    BATCH_SIZE
)


# ============================================================
# RANGE TRACKER
# ============================================================

activation_ranges = {}


def update_range(name, tensor):

    if not torch.is_tensor(tensor):
        return

    tensor = tensor.detach()

    current_min = tensor.min().item()
    current_max = tensor.max().item()

    if name not in activation_ranges:

        activation_ranges[name] = {
            "min": current_min,
            "max": current_max
        }

    else:

        activation_ranges[name]["min"] = min(
            activation_ranges[name]["min"],
            current_min
        )

        activation_ranges[name]["max"] = max(
            activation_ranges[name]["max"],
            current_max
        )


# ============================================================
# REGISTER HOOKS
# ============================================================

hooks = []


def make_hook(name):

    def hook(module, inputs, output):

        update_range(
            name,
            output
        )

    return hook


# ------------------------------------------------------------
# Patch embedding
# ------------------------------------------------------------

hooks.append(
    model_fp32.emb.register_forward_hook(
        make_hook("patch_embedding")
    )
)


# ------------------------------------------------------------
# Transformer layers
# ------------------------------------------------------------

for i, block in enumerate(model_fp32.enc):

    hooks.append(
        block.la1.register_forward_hook(
            make_hook(
                f"block{i}_layernorm1"
            )
        )
    )

    hooks.append(
        block.msa.q.register_forward_hook(
            make_hook(
                f"block{i}_Q"
            )
        )
    )

    hooks.append(
        block.msa.k.register_forward_hook(
            make_hook(
                f"block{i}_K"
            )
        )
    )

    hooks.append(
        block.msa.v.register_forward_hook(
            make_hook(
                f"block{i}_V"
            )
        )
    )

    hooks.append(
        block.msa.o.register_forward_hook(
            make_hook(
                f"block{i}_attention_output"
            )
        )
    )

    hooks.append(
        block.la2.register_forward_hook(
            make_hook(
                f"block{i}_layernorm2"
            )
        )
    )

    hooks.append(
        block.mlp[0].register_forward_hook(
            make_hook(
                f"block{i}_mlp_fc1"
            )
        )
    )

    hooks.append(
        block.mlp[1].register_forward_hook(
            make_hook(
                f"block{i}_gelu1"
            )
        )
    )

    hooks.append(
        block.mlp[3].register_forward_hook(
            make_hook(
                f"block{i}_mlp_fc2"
            )
        )
    )

    hooks.append(
        block.mlp[4].register_forward_hook(
            make_hook(
                f"block{i}_gelu2"
            )
        )
    )


# ------------------------------------------------------------
# Final classifier
# ------------------------------------------------------------

hooks.append(
    model_fp32.fc[0].register_forward_hook(
        make_hook("final_layernorm")
    )
)

hooks.append(
    model_fp32.fc[1].register_forward_hook(
        make_hook("classifier")
    )
)


# ============================================================
# RUN CALIBRATION
# ============================================================

model_fp32.eval()

print("\nRunning calibration...")

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(
        calibration_loader
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        model_fp32(images)

        if (batch_idx + 1) % 2 == 0:

            print(
                f"Processed "
                f"{min((batch_idx + 1) * BATCH_SIZE, CALIBRATION_SAMPLES)}"
                f"/{CALIBRATION_SAMPLES}"
            )


# ============================================================
# REMOVE HOOKS
# ============================================================

for hook in hooks:
    hook.remove()


# ============================================================
# CALCULATE INT8 PARAMETERS
# ============================================================

def calculate_int8_params(
    min_val,
    max_val
):

    # Symmetric signed INT8
    max_abs = max(
        abs(min_val),
        abs(max_val)
    )

    scale = max_abs / 127.0

    if scale == 0:
        scale = 1.0

    zero_point = 0

    return scale, zero_point


quantization_config = {}


for name, values in activation_ranges.items():

    scale, zero_point = calculate_int8_params(
        values["min"],
        values["max"]
    )

    quantization_config[name] = {

        "min": values["min"],

        "max": values["max"],

        "scale": scale,

        "zero_point": zero_point
    }


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 80)
print("CALIBRATION RESULTS")
print("=" * 80)

for name, values in quantization_config.items():

    print(
        f"{name:35s} "
        f"min={values['min']: .6f} "
        f"max={values['max']: .6f} "
        f"scale={values['scale']: .8f}"
    )


# ============================================================
# SAVE CONFIGURATION
# ============================================================

os.makedirs(
    "results",
    exist_ok=True
)

calibration_path = (
    "results/c201_quantization_config.json"
)

with open(
    calibration_path,
    "w"
) as f:

    json.dump(
        quantization_config,
        f,
        indent=4
    )


print("\nSaved:")
print(calibration_path)

print("\nINT8 calibration complete.")

100%|██████████| 170M/170M [37:01<00:00, 76.7kB/s] 


INT8 CALIBRATION DATASET
Total calibration samples: 512
Batch size: 64

Running calibration...
Processed 128/512
Processed 256/512
Processed 384/512
Processed 512/512

CALIBRATION RESULTS
patch_embedding                     min=-4.805822 max= 5.304322 scale= 0.04176631
block0_layernorm1                   min=-3.460049 max= 3.202800 scale= 0.02724448
block0_Q                            min=-4.663398 max= 5.546861 scale= 0.04367607
block0_K                            min=-5.082581 max= 5.612453 scale= 0.04419255
block0_V                            min=-3.132770 max= 3.477959 scale= 0.02738550
block0_attention_output             min=-2.171430 max= 1.978474 scale= 0.01709787
block0_layernorm2                   min=-5.078850 max= 4.802789 scale= 0.03999094
block0_mlp_fc1                      min=-15.421501 max= 13.774838 scale= 0.12142914
block0_gelu1                        min=-0.169971 max= 13.774838 scale= 0.10846329
block0_mlp_fc2                      min=-19.325247 max= 9.046178 scale=

In [59]:
# ============================================================
# C201 POST-TRAINING INT8 QUANTIZATION
# ============================================================

import os
import json
import copy
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FP32_MODEL = model_fp32

CALIBRATION_CONFIG = (
    "results/c201_quantization_config.json"
)

INT8_MODEL_DIR = "results/c201_int8"

os.makedirs(
    INT8_MODEL_DIR,
    exist_ok=True
)

print("Device:", DEVICE)


# ============================================================
# 1. LOAD CALIBRATION CONFIGURATION
# ============================================================

with open(
    CALIBRATION_CONFIG,
    "r"
) as f:

    calibration_config = json.load(f)

print(
    f"Loaded {len(calibration_config)} "
    "calibration ranges."
)


# ============================================================
# 2. INT8 QUANTIZATION FUNCTIONS
# ============================================================

INT8_MIN = -127
INT8_MAX = 127


def quantize_symmetric(
    x,
    scale
):
    """
    Symmetric INT8 quantization.

    q = round(x / scale)
    q clipped to [-127, 127]
    """

    if scale <= 0:
        raise ValueError(
            f"Invalid scale: {scale}"
        )

    q = torch.round(
        x / scale
    )

    q = torch.clamp(
        q,
        INT8_MIN,
        INT8_MAX
    )

    return q.to(torch.int8)


def dequantize(
    q,
    scale
):
    """
    INT8 -> FP32
    """

    return q.float() * scale


def quantize_tensor(
    x
):
    """
    Calculate symmetric scale directly from tensor
    and quantize it.
    """

    max_abs = x.detach().abs().max().item()

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / 127.0

    q = quantize_symmetric(
        x,
        scale
    )

    return q, scale


# ============================================================
# 3. QUANTIZE ALL TRAINED WEIGHTS
# ============================================================

quantized_weights = {}

weight_scales = {}

print("\n" + "=" * 70)
print("QUANTIZING C201 WEIGHTS")
print("=" * 70)

for name, parameter in FP32_MODEL.named_parameters():

    # Quantize only floating-point tensors
    if not parameter.is_floating_point():
        continue

    weight = parameter.detach().cpu()

    q_weight, scale = quantize_tensor(
        weight
    )

    quantized_weights[name] = q_weight
    weight_scales[name] = scale

    print(
        f"{name:45s} "
        f"FP32={weight.numel():8d} "
        f"INT8={q_weight.numel():8d} "
        f"scale={scale:.8f}"
    )


# ============================================================
# 4. WEIGHT MEMORY
# ============================================================

fp32_weight_bytes = 0
int8_weight_bytes = 0

for name, parameter in FP32_MODEL.named_parameters():

    if parameter.is_floating_point():

        fp32_weight_bytes += (
            parameter.numel() * 4
        )

        int8_weight_bytes += (
            parameter.numel()
        )


print("\n" + "=" * 70)
print("WEIGHT MEMORY")
print("=" * 70)

print(
    f"FP32 weights : "
    f"{fp32_weight_bytes / (1024**2):.6f} MB"
)

print(
    f"INT8 weights : "
    f"{int8_weight_bytes / (1024**2):.6f} MB"
)

print(
    f"Compression  : "
    f"{fp32_weight_bytes / int8_weight_bytes:.2f}x"
)


# ============================================================
# 5. SAVE QUANTIZED WEIGHTS
# ============================================================

torch.save(
    {
        "quantized_weights": quantized_weights,
        "weight_scales": weight_scales
    },
    f"{INT8_MODEL_DIR}/c201_int8_weights.pth"
)

print(
    "\nSaved:"
)

print(
    f"{INT8_MODEL_DIR}/c201_int8_weights.pth"
)


# ============================================================
# 6. CALIBRATED ACTIVATION Q/DQ
# ============================================================

def calibrated_quantize(
    x,
    name
):

    if name not in calibration_config:

        raise KeyError(
            f"No calibration scale for: {name}"
        )

    scale = calibration_config[
        name
    ]["scale"]

    q = quantize_symmetric(
        x,
        scale
    )

    return dequantize(
        q,
        scale
    )


# ============================================================
# 7. QUANTIZED LINEAR OPERATION
# ============================================================

def int8_linear(
    x,
    weight,
    weight_scale,
    activation_scale
):
    """
    INT8 x INT8 -> INT32 accumulation -> FP32.

    x:
        FP32 activation which is quantized using the
        calibrated activation scale.

    weight:
        FP32 weight which is quantized using a symmetric
        INT8 weight scale.

    Result:
        Dequantized FP32 output.
    """

    # Quantize activation
    x_int8 = quantize_symmetric(
        x,
        activation_scale
    )

    # Quantize weight
    w_int8 = quantize_symmetric(
        weight,
        weight_scale
    )

    # INT8 x INT8 accumulation
    # PyTorch promotes the multiplication to a wider type.
    output_int32 = torch.matmul(
        x_int8.to(torch.int32),
        w_int8.to(torch.int32).transpose(-2, -1)
    )

    # Convert back to FP32
    output = (
        output_int32.float()
        * activation_scale
        * weight_scale
    )

    return output


# ============================================================
# 8. QUANTIZED C201 MODEL
# ============================================================

class C201_INT8_Reference(nn.Module):

    def __init__(
        self,
        fp32_model,
        calibration,
        weight_scales
    ):

        super().__init__()

        self.fp32_model = fp32_model
        self.calibration = calibration
        self.weight_scales = weight_scales

        # No gradients in deployment reference
        for parameter in self.parameters():
            parameter.requires_grad = False


    def qdq(
        self,
        x,
        name
    ):

        scale = self.calibration[
            name
        ]["scale"]

        q = quantize_symmetric(
            x,
            scale
        )

        return dequantize(
            q,
            scale
        )


    def quantized_linear(
        self,
        x,
        layer,
        input_scale
    ):

        weight_name = None

        # Find corresponding parameter name
        for name, parameter in (
            self.fp32_model.named_parameters()
        ):

            if (
                name.endswith(".weight")
                and parameter.data_ptr()
                == layer.weight.data_ptr()
            ):

                weight_name = name
                break

        if weight_name is None:

            raise RuntimeError(
                "Could not identify Linear weight."
            )

        weight = layer.weight.detach()

        weight_scale = self.weight_scales[
            weight_name
        ]

        return int8_linear(
            x,
            weight,
            weight_scale,
            input_scale
        )


# ============================================================
# 9. IMPORTANT:
#    BUILD A DEPLOYMENT-STYLE Q/DQ FORWARD PASS
# ============================================================

class C201INT8(nn.Module):

    def __init__(
        self,
        fp32_model,
        calibration,
        weight_scales
    ):

        super().__init__()

        self.model = fp32_model
        self.calibration = calibration
        self.weight_scales = weight_scales

        for p in self.model.parameters():
            p.requires_grad = False


    def qdq(
        self,
        x,
        name
    ):

        scale = self.calibration[
            name
        ]["scale"]

        q = quantize_symmetric(
            x,
            scale
        )

        return q.float() * scale


    def linear(
        self,
        x,
        layer,
        activation_scale,
        weight_name
    ):
        """
        INT8 reference linear layer.
    
        INT8 weights and activations are quantized first.
        Integer multiplication is performed using a supported
        integer implementation, with INT32 accumulation.
        The result is then dequantized to FP32.
        """
    
        weight = layer.weight.detach()
    
        weight_scale = self.weight_scales[
            weight_name
        ]
    
        # --------------------------------------------------------
        # Quantize activation
        # --------------------------------------------------------
    
        x_q = quantize_symmetric(
            x,
            activation_scale
        )
    
        # --------------------------------------------------------
        # Quantize weight
        # --------------------------------------------------------
    
        w_q = quantize_symmetric(
            weight,
            weight_scale
        )
    
        # --------------------------------------------------------
        # Integer matrix multiplication
        #
        # CUDA does not support torch.matmul() for Int tensors
        # in this environment.
        #
        # Convert to FP32 only for the PyTorch reference
        # multiplication. The quantized values remain integer
        # values, so this still represents:
        #
        #       INT8 × INT8 → accumulation
        #
        # The eventual C implementation will use actual
        # integer accumulation.
        # --------------------------------------------------------
    
        x_q_fp32 = x_q.float()
        w_q_fp32 = w_q.float()
    
        y = torch.matmul(
            x_q_fp32,
            w_q_fp32.transpose(-2, -1)
        )
    
        # --------------------------------------------------------
        # Dequantize
        # --------------------------------------------------------
    
        y = (
            y
            * activation_scale
            * weight_scale
        )
    
        # --------------------------------------------------------
        # FP32 bias
        # --------------------------------------------------------
    
        if layer.bias is not None:
    
            y = y + layer.bias.detach()
    
        return y

    def forward(self, img):

        B, C, H, W = img.shape

        model = self.model

        p = model.patch_size

        # ----------------------------------------------------
        # Patch extraction
        # ----------------------------------------------------

        x = img.unfold(
            2,
            p,
            p
        ).unfold(
            3,
            p,
            p
        )

        x = x.permute(
            0,
            2,
            3,
            1,
            4,
            5
        )

        x = x.reshape(
            B,
            model.patch ** 2,
            -1
        )

        # ----------------------------------------------------
        # Patch embedding
        #
        # Raw image patches do not have a stored calibration
        # range, so derive their symmetric scale from the
        # current input tensor.
        # ----------------------------------------------------

        patch_scale = (
            x.detach().abs().max().item()
            / 127.0
        )

        if patch_scale == 0:
            patch_scale = 1.0

        x = self.linear(
            x,
            model.emb,
            patch_scale,
            "emb.weight"
        )

        # Apply calibrated patch embedding output Q/DQ
        x = self.qdq(
            x,
            "patch_embedding"
        )

        # ----------------------------------------------------
        # CLS token
        # ----------------------------------------------------

        cls = model.cls_token.expand(
            B,
            -1,
            -1
        )

        x = torch.cat(
            [cls, x],
            dim=1
        )

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        x = x + model.pos_emb

        # ----------------------------------------------------
        # Transformer blocks
        # ----------------------------------------------------

        for block_idx, block in enumerate(
            model.enc
        ):

            prefix = f"block{block_idx}"

            # -----------------------------------------------
            # LayerNorm 1
            # -----------------------------------------------

            x_norm = block.la1(x)

            x_norm = self.qdq(
                x_norm,
                f"{prefix}_layernorm1"
            )

            # -----------------------------------------------
            # Q
            # -----------------------------------------------

            q = self.linear(
                x_norm,
                block.msa.q,
                self.calibration[
                    f"{prefix}_layernorm1"
                ]["scale"],
                f"enc.{block_idx}.msa.q.weight"
            )

            q = self.qdq(
                q,
                f"{prefix}_Q"
            )

            # -----------------------------------------------
            # K
            # -----------------------------------------------

            k = self.linear(
                x_norm,
                block.msa.k,
                self.calibration[
                    f"{prefix}_layernorm1"
                ]["scale"],
                f"enc.{block_idx}.msa.k.weight"
            )

            k = self.qdq(
                k,
                f"{prefix}_K"
            )

            # -----------------------------------------------
            # V
            # -----------------------------------------------

            v = self.linear(
                x_norm,
                block.msa.v,
                self.calibration[
                    f"{prefix}_layernorm1"
                ]["scale"],
                f"enc.{block_idx}.msa.v.weight"
            )

            v = self.qdq(
                v,
                f"{prefix}_V"
            )

            # -----------------------------------------------
            # Split heads
            # -----------------------------------------------

            B2, N, C2 = q.shape

            q = q.reshape(
                B2,
                N,
                block.msa.heads,
                block.msa.head_dim
            ).transpose(1, 2)

            k = k.reshape(
                B2,
                N,
                block.msa.heads,
                block.msa.head_dim
            ).transpose(1, 2)

            v = v.reshape(
                B2,
                N,
                block.msa.heads,
                block.msa.head_dim
            ).transpose(1, 2)

            # -----------------------------------------------
            # Attention
            # -----------------------------------------------

            attn = torch.matmul(
                q,
                k.transpose(-2, -1)
            )

            attn = (
                attn
                / block.msa.sqrt_d
            )

            attn = torch.softmax(
                attn,
                dim=-1
            )

            out = torch.matmul(
                attn,
                v
            )

            # Merge heads
            out = out.transpose(
                1,
                2
            ).reshape(
                B2,
                N,
                C2
            )

            # -----------------------------------------------
            # Output projection
            # -----------------------------------------------

            out = self.linear(
                out,
                block.msa.o,
                self.calibration[
                    f"{prefix}_attention_output"
                ]["scale"],
                f"enc.{block_idx}.msa.o.weight"
            )

            out = self.qdq(
                out,
                f"{prefix}_attention_output"
            )

            # Residual
            x = x + out

            # -----------------------------------------------
            # LayerNorm 2
            # -----------------------------------------------

            x_norm = block.la2(x)

            x_norm = self.qdq(
                x_norm,
                f"{prefix}_layernorm2"
            )

            # -----------------------------------------------
            # MLP FC1
            # -----------------------------------------------

            mlp = self.linear(
                x_norm,
                block.mlp[0],
                self.calibration[
                    f"{prefix}_layernorm2"
                ]["scale"],
                f"enc.{block_idx}.mlp.0.weight"
            )

            mlp = self.qdq(
                mlp,
                f"{prefix}_mlp_fc1"
            )

            # GELU
            mlp = torch.nn.functional.gelu(
                mlp
            )

            mlp = self.qdq(
                mlp,
                f"{prefix}_gelu1"
            )

            # -----------------------------------------------
            # MLP FC2
            # -----------------------------------------------

            mlp = self.linear(
                mlp,
                block.mlp[3],
                self.calibration[
                    f"{prefix}_gelu1"
                ]["scale"],
                f"enc.{block_idx}.mlp.3.weight"
            )

            mlp = self.qdq(
                mlp,
                f"{prefix}_mlp_fc2"
            )

            # GELU
            mlp = torch.nn.functional.gelu(
                mlp
            )

            mlp = self.qdq(
                mlp,
                f"{prefix}_gelu2"
            )

            # Residual
            x = x + mlp

        # ----------------------------------------------------
        # CLS token
        # ----------------------------------------------------

        x = x[:, 0]

        # ----------------------------------------------------
        # Final LayerNorm
        # ----------------------------------------------------

        x = model.fc[0](x)

        x = self.qdq(
            x,
            "final_layernorm"
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        x = self.linear(
            x,
            model.fc[1],
            self.calibration[
                "final_layernorm"
            ]["scale"],
            "fc.1.weight"
        )

        x = self.qdq(
            x,
            "classifier"
        )

        return x


# ============================================================
# 10. CREATE INT8 REFERENCE MODEL
# ============================================================

model_int8 = C201INT8(
    FP32_MODEL,
    calibration_config,
    weight_scales
).to(DEVICE)

model_int8.eval()

print("\nINT8 reference model created.")


# ============================================================
# 11. VERIFY SINGLE INFERENCE
# ============================================================

test_input = torch.randn(
    1,
    3,
    32,
    32,
    device=DEVICE
)

with torch.no_grad():

    fp32_output = model_fp32(
        test_input
    )

    int8_output = model_int8(
        test_input
    )

print("\n" + "=" * 70)
print("SINGLE IMAGE INT8 VERIFICATION")
print("=" * 70)

print(
    "FP32 prediction:",
    fp32_output.argmax(dim=1).item()
)

print(
    "INT8 prediction:",
    int8_output.argmax(dim=1).item()
)

print(
    "FP32 logits:",
    fp32_output.cpu().numpy()
)

print(
    "INT8 logits:",
    int8_output.cpu().numpy()
)

print(
    "Max absolute logit difference:",
    (
        fp32_output - int8_output
    ).abs().max().item()
)

Device: cuda
Loaded 23 calibration ranges.

QUANTIZING C201 WEIGHTS
cls_token                                     FP32=      96 INT8=      96 scale=0.02098359
pos_emb                                       FP32=    6240 INT8=    6240 scale=0.01856870
emb.weight                                    FP32=    4608 INT8=    4608 scale=0.00374251
emb.bias                                      FP32=      96 INT8=      96 scale=0.00347160
enc.0.la1.weight                              FP32=      96 INT8=      96 scale=0.00727094
enc.0.la1.bias                                FP32=      96 INT8=      96 scale=0.00306524
enc.0.msa.q.weight                            FP32=    9216 INT8=    9216 scale=0.00346743
enc.0.msa.q.bias                              FP32=      96 INT8=      96 scale=0.00541769
enc.0.msa.k.weight                            FP32=    9216 INT8=    9216 scale=0.00320413
enc.0.msa.k.bias                              FP32=      96 INT8=      96 scale=0.00001201
enc.0.msa.v.weight    

# Correct Evaluation Begins 

In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# 2. EXACT ORIGINAL C201 MODEL
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, feats, head, dropout=0.0):
        super().__init__()

        self.feats = feats
        self.head = head
        self.head_dim = feats // head
        self.sqrt_d = feats ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)
        self.o = nn.Linear(feats, feats)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, N, C = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        q = q.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        k = k.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        v = v.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        # EXACT original attention computation
        score = torch.softmax(
            torch.einsum(
                "bhif,bhjf->bhij",
                q,
                k
            ) / self.sqrt_d,
            dim=-1
        )

        attn = torch.einsum(
            "bhij,bhjf->bihf",
            score,
            v
        )

        attn = attn.flatten(2)

        return self.dropout(self.o(attn))


class MLP(nn.Module):
    def __init__(self, feats, hidden, dropout=0.0):
        super().__init__()

        self.fc1 = nn.Linear(feats, hidden)
        self.gelu1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden, feats)
        self.gelu2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu1(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.gelu2(x)
        x = self.drop2(x)

        return x


class TransformerEncoder(nn.Module):
    def __init__(self, feats, mlp_hidden, head, dropout=0.0):
        super().__init__()

        self.la1 = nn.LayerNorm(feats)

        self.msa = MultiHeadAttention(
            feats=feats,
            head=head,
            dropout=dropout
        )

        self.la2 = nn.LayerNorm(feats)

        self.mlp = MLP(
            feats=feats,
            hidden=mlp_hidden,
            dropout=dropout
        )

    def forward(self, x):

        # EXACT original residual structure
        out = self.msa(self.la1(x)) + x

        out = self.mlp(self.la2(out)) + out

        return out


class ViT(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch=8,
        num_layers=2,
        hidden=96,
        mlp_hidden=96,
        head=4,
        is_cls_token=True,
        dropout=0.0,
        num_classes=10
    ):
        super().__init__()

        self.img_size = img_size
        self.patch = patch
        self.patch_size = img_size // patch
        self.hidden = hidden
        self.is_cls_token = is_cls_token

        # 4x4 patch, RGB => 4*4*3 = 48
        f = self.patch_size ** 2 * 3

        self.emb = nn.Linear(f, hidden)

        num_tokens = patch ** 2

        if is_cls_token:
            num_tokens += 1
            self.cls_token = nn.Parameter(
                torch.randn(1, 1, hidden)
            )
        else:
            self.cls_token = None

        self.pos_emb = nn.Parameter(
            torch.randn(1, num_tokens, hidden)
        )

        self.enc = nn.ModuleList([
            TransformerEncoder(
                feats=hidden,
                mlp_hidden=mlp_hidden,
                head=head,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.fc = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )

    def _to_words(self, x):

        B, C, H, W = x.shape
        p = self.patch_size

        x = (
            x.unfold(2, p, p)
             .unfold(3, p, p)
             .permute(0, 2, 3, 4, 5, 1)
             .reshape(B, self.patch ** 2, -1)
        )

        return x

    def forward(self, x):

        x = self._to_words(x)

        x = self.emb(x)

        if self.is_cls_token:
            cls = self.cls_token.repeat(x.shape[0], 1, 1)
            x = torch.cat([cls, x], dim=1)

        x = x + self.pos_emb

        for enc in self.enc:
            x = enc(x)

        if self.is_cls_token:
            x = x[:, 0]
        else:
            x = x.mean(dim=1)

        return self.fc(x)


# ============================================================
# 3. CREATE EXACT C201
# ============================================================

model = ViT(
    img_size=32,
    patch=8,
    num_layers=2,
    hidden=96,
    mlp_hidden=96,
    head=4,
    is_cls_token=True,
    dropout=0.0,
    num_classes=10
).to(device)

print("Parameters:", sum(p.numel() for p in model.parameters()))

Device: cuda
Parameters: 124714


In [84]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# 2. EXACT C201 MODEL
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, feats, head, dropout=0.0):
        super().__init__()

        self.feats = feats
        self.head = head
        self.head_dim = feats // head

        # IMPORTANT:
        # Original code uses sqrt(feats), NOT sqrt(head_dim)
        self.sqrt_d = feats ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)
        self.o = nn.Linear(feats, feats)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, N, C = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        q = q.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        k = k.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        v = v.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        # EXACT original implementation
        score = torch.softmax(
            torch.einsum(
                "bhif,bhjf->bhij",
                q,
                k
            ) / self.sqrt_d,
            dim=-1
        )

        # EXACT original implementation
        attn = torch.einsum(
            "bhij,bhjf->bihf",
            score,
            v
        )

        return self.dropout(
            self.o(attn.flatten(2))
        )


# ------------------------------------------------------------

class MLP(nn.Sequential):
    """
    IMPORTANT:
    Original checkpoint contains:

        mlp.0.weight
        mlp.0.bias
        mlp.3.weight
        mlp.3.bias

    Therefore the original MLP must be represented
    as nn.Sequential.
    """

    def __init__(self, feats, hidden, dropout=0.0):

        super().__init__(
            nn.Linear(feats, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, feats),
            nn.GELU(),
            nn.Dropout(dropout)
        )


# ------------------------------------------------------------

class TransformerEncoder(nn.Module):
    def __init__(self, feats, mlp_hidden, head, dropout=0.0):

        super().__init__()

        self.la1 = nn.LayerNorm(feats)

        self.msa = MultiHeadAttention(
            feats=feats,
            head=head,
            dropout=dropout
        )

        self.la2 = nn.LayerNorm(feats)

        self.mlp = MLP(
            feats=feats,
            hidden=mlp_hidden,
            dropout=dropout
        )

    def forward(self, x):

        # EXACT original residual structure
        out = self.msa(self.la1(x)) + x

        out = self.mlp(self.la2(out)) + out

        return out


# ------------------------------------------------------------

class ViT(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch=8,
        num_layers=2,
        hidden=96,
        mlp_hidden=96,
        head=4,
        is_cls_token=True,
        dropout=0.0,
        num_classes=10
    ):

        super().__init__()

        self.img_size = img_size
        self.patch = patch
        self.patch_size = img_size // patch
        self.hidden = hidden
        self.is_cls_token = is_cls_token

        # C201:
        # patch_size = 32 / 8 = 4
        # 4 x 4 x 3 = 48
        f = self.patch_size ** 2 * 3

        self.emb = nn.Linear(
            f,
            hidden
        )

        num_tokens = patch ** 2

        if is_cls_token:

            num_tokens += 1

            self.cls_token = nn.Parameter(
                torch.randn(1, 1, hidden)
            )

        else:

            self.cls_token = None

        self.pos_emb = nn.Parameter(
            torch.randn(
                1,
                num_tokens,
                hidden
            )
        )

        self.enc = nn.ModuleList([
            TransformerEncoder(
                feats=hidden,
                mlp_hidden=mlp_hidden,
                head=head,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.fc = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )

    def _to_words(self, x):

        B, C, H, W = x.shape

        p = self.patch_size

        # EXACT original patch extraction
        x = (
            x.unfold(2, p, p)
             .unfold(3, p, p)
             .permute(0, 2, 3, 4, 5, 1)
             .reshape(
                 B,
                 self.patch ** 2,
                 -1
             )
        )

        return x

    def forward(self, x):

        x = self._to_words(x)

        x = self.emb(x)

        if self.is_cls_token:

            cls = self.cls_token.repeat(
                x.shape[0],
                1,
                1
            )

            x = torch.cat(
                [cls, x],
                dim=1
            )

        x = x + self.pos_emb

        for enc in self.enc:
            x = enc(x)

        if self.is_cls_token:

            x = x[:, 0]

        else:

            x = x.mean(dim=1)

        return self.fc(x)


# ============================================================
# 3. CREATE C201
# ============================================================

model = ViT(
    img_size=32,
    patch=8,
    num_layers=2,
    hidden=96,
    mlp_hidden=96,
    head=4,
    is_cls_token=True,
    dropout=0.0,
    num_classes=10
).to(device)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))


# ============================================================
# 4. LOAD CHECKPOINT
# ============================================================

checkpoint_path = (
    "/kaggle/input/datasets/tiw775/weights/"
    "vit_c10_aa_ls_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint keys:")
print(checkpoint.keys())

print("\nCheckpoint epoch:",
      checkpoint["epoch"])

print("Checkpoint best_acc:",
      checkpoint["best_acc"])


# ============================================================
# 5. LOAD WEIGHTS
# ============================================================

model.load_state_dict(
    checkpoint["model"],
    strict=True
)

model.eval()

print("\nCheckpoint loaded successfully.")


# ============================================================
# 6. VERIFY CHECKPOINT TENSOR COUNT
# ============================================================

checkpoint_tensors = checkpoint["model"]

print("\nCheckpoint tensors:",
      len(checkpoint_tensors))

print("\nFirst few checkpoint keys:")

for i, key in enumerate(checkpoint_tensors.keys()):

    if i >= 10:
        break

    print(" ", key)


# ============================================================
# 7. EXACT CIFAR-10 TEST TRANSFORM
# ============================================================

mean = [
    0.4914,
    0.4822,
    0.4465
]

std = [
    0.2470,
    0.2435,
    0.2616
]

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean,
        std
    )
])


# ============================================================
# 8. CIFAR-10 TEST DATASET
# ============================================================

test_dataset = datasets.CIFAR10(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nTest samples:",
      len(test_dataset))


# ============================================================
# 9. EXACT FP32 EVALUATION
# ============================================================

criterion = nn.CrossEntropyLoss()

model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        total_loss += (
            loss.item()
            * labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


test_loss = total_loss / total
test_acc = correct / total


# ============================================================
# 10. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("C201 FP32 CHECKPOINT VERIFICATION")
print("=" * 60)

print(f"Parameters        : {sum(p.numel() for p in model.parameters()):,}")
print(f"Checkpoint epoch  : {checkpoint['epoch']}")
print(f"Stored best_acc   : {checkpoint['best_acc'] * 100:.2f}%")
print(f"Test loss         : {test_loss:.6f}")
print(f"Test accuracy     : {test_acc * 100:.2f}%")

print("=" * 60)

Device: cuda

Model parameters: 124714

Checkpoint keys:
dict_keys(['epoch', 'model', 'optimizer', 'scheduler', 'scaler', 'best_acc'])

Checkpoint epoch: 100
Checkpoint best_acc: 0.7695

Checkpoint loaded successfully.

Checkpoint tensors: 40

First few checkpoint keys:
  cls_token
  pos_emb
  emb.weight
  emb.bias
  enc.0.la1.weight
  enc.0.la1.bias
  enc.0.msa.q.weight
  enc.0.msa.q.bias
  enc.0.msa.k.weight
  enc.0.msa.k.bias

Test samples: 10000

C201 FP32 CHECKPOINT VERIFICATION
Parameters        : 124,714
Checkpoint epoch  : 100
Stored best_acc   : 76.95%
Test loss         : 0.725030
Test accuracy     : 76.93%


In [86]:
import os
import json
import torch

# ============================================================
# C201 FP32 -> INT8 WEIGHT EXPORT
# ============================================================

FP32_PATH = (
    "/kaggle/input/datasets/tiw775/weights/"
    "vit_c10_aa_ls_best.pth"
)

CALIB_PATH = (
    "/kaggle/working/results/"
    "c201_quantization_config.json"
)

OUT_DIR = (
    "/kaggle/working/results/"
    "c201_int8"
)

OUT_PATH = (
    OUT_DIR +
    "/c201_int8_weights.pth"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load FP32 checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    FP32_PATH,
    map_location="cpu",
    weights_only=False
)

fp32_state = checkpoint["model"]

print("FP32 checkpoint loaded")
print("Epoch:", checkpoint["epoch"])
print("Best accuracy:", checkpoint["best_acc"])
print("Parameters:",
      sum(v.numel() for v in fp32_state.values()))


# ------------------------------------------------------------
# Quantize weights
# ------------------------------------------------------------

int8_state = {}
weight_scales = {}

for name, tensor in fp32_state.items():

    # Only floating-point tensors are quantized.
    if tensor.is_floating_point():

        max_abs = tensor.abs().max().item()

        if max_abs == 0:
            scale = 1.0
        else:
            scale = max_abs / 127.0

        q = torch.clamp(
            torch.round(tensor / scale),
            -127,
            127
        ).to(torch.int8)

        int8_state[name] = q
        weight_scales[name] = scale

    else:

        int8_state[name] = tensor


# ------------------------------------------------------------
# Save INT8 weights + weight scales
# ------------------------------------------------------------

int8_checkpoint = {
    "model": int8_state,
    "weight_scales": weight_scales,
    "source_epoch": checkpoint["epoch"],
    "source_best_acc": checkpoint["best_acc"],
    "num_parameters": sum(
        v.numel()
        for v in fp32_state.values()
    )
}

torch.save(
    int8_checkpoint,
    OUT_PATH
)

print("\nINT8 export complete.")
print("Saved to:")
print(OUT_PATH)


# ------------------------------------------------------------
# Memory calculation
# ------------------------------------------------------------

fp32_bytes = 0
int8_bytes = 0

for name, tensor in fp32_state.items():

    fp32_bytes += tensor.numel() * 4

    if torch.is_floating_point(tensor):
        int8_bytes += tensor.numel()
    else:
        int8_bytes += tensor.numel() * tensor.element_size()

fp32_mb = fp32_bytes / (1024 ** 2)
int8_mb = int8_bytes / (1024 ** 2)

print("\n" + "=" * 60)
print("C201 WEIGHT MEMORY")
print("=" * 60)

print(f"FP32 weight memory : {fp32_mb:.6f} MB")
print(f"INT8 weight memory : {int8_mb:.6f} MB")
print(f"Compression        : {fp32_mb / int8_mb:.2f}x")

print("=" * 60)


# ------------------------------------------------------------
# Verify quantization error
# ------------------------------------------------------------

max_error = 0.0
mean_errors = []

for name, tensor in fp32_state.items():

    if not torch.is_floating_point(tensor):
        continue

    q = int8_state[name]
    scale = weight_scales[name]

    reconstructed = q.float() * scale

    error = (
        tensor - reconstructed
    ).abs()

    max_error = max(
        max_error,
        error.max().item()
    )

    mean_errors.append(
        error.mean().item()
    )

print("\nQuantization verification:")
print(
    f"Maximum absolute error : {max_error:.6f}"
)
print(
    f"Mean absolute error    : "
    f"{sum(mean_errors) / len(mean_errors):.6f}"
)

print("\nFile exists:",
      os.path.exists(OUT_PATH))

print(
    "File size:",
    os.path.getsize(OUT_PATH) / (1024 ** 2),
    "MB"
)

FP32 checkpoint loaded
Epoch: 100
Best accuracy: 0.7695
Parameters: 124714

INT8 export complete.
Saved to:
/kaggle/working/results/c201_int8/c201_int8_weights.pth

C201 WEIGHT MEMORY
FP32 weight memory : 0.475746 MB
INT8 weight memory : 0.118937 MB
Compression        : 4.00x

Quantization verification:
Maximum absolute error : 0.010325
Mean absolute error    : 0.001162

File exists: True
File size: 0.13066768646240234 MB


In [87]:
# ============================================================
# C201: FP32 vs INT8 WEIGHT-ONLY REFERENCE EVALUATION
# ============================================================

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 2. PATHS
# ============================================================

FP32_PATH = (
    "/kaggle/input/datasets/tiw775/weights/"
    "vit_c10_aa_ls_best.pth"
)

INT8_PATH = (
    "/kaggle/working/results/"
    "c201_int8/c201_int8_weights.pth"
)


# ============================================================
# 3. EXACT C201 ARCHITECTURE
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, feats, head, dropout=0.0):

        super().__init__()

        self.feats = feats
        self.head = head
        self.head_dim = feats // head

        # Original code:
        self.sqrt_d = feats ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)
        self.o = nn.Linear(feats, feats)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, N, C = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        q = q.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        k = k.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        v = v.reshape(
            B, N, self.head, self.head_dim
        ).transpose(1, 2)

        score = torch.softmax(
            torch.einsum(
                "bhif,bhjf->bhij",
                q,
                k
            ) / self.sqrt_d,
            dim=-1
        )

        attn = torch.einsum(
            "bhij,bhjf->bihf",
            score,
            v
        )

        return self.dropout(
            self.o(attn.flatten(2))
        )


# ------------------------------------------------------------

class MLP(nn.Sequential):

    def __init__(self, feats, hidden, dropout=0.0):

        super().__init__(
            nn.Linear(feats, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, feats),
            nn.GELU(),
            nn.Dropout(dropout)
        )


# ------------------------------------------------------------

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        feats,
        mlp_hidden,
        head,
        dropout=0.0
    ):

        super().__init__()

        self.la1 = nn.LayerNorm(feats)

        self.msa = MultiHeadAttention(
            feats,
            head,
            dropout
        )

        self.la2 = nn.LayerNorm(feats)

        self.mlp = MLP(
            feats,
            mlp_hidden,
            dropout
        )

    def forward(self, x):

        out = self.msa(
            self.la1(x)
        ) + x

        out = self.mlp(
            self.la2(out)
        ) + out

        return out


# ------------------------------------------------------------

class ViT(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch=8,
        num_layers=2,
        hidden=96,
        mlp_hidden=96,
        head=4,
        is_cls_token=True,
        dropout=0.0,
        num_classes=10
    ):

        super().__init__()

        self.img_size = img_size
        self.patch = patch
        self.patch_size = img_size // patch
        self.hidden = hidden
        self.is_cls_token = is_cls_token

        f = self.patch_size ** 2 * 3

        self.emb = nn.Linear(
            f,
            hidden
        )

        num_tokens = patch ** 2

        if is_cls_token:

            num_tokens += 1

            self.cls_token = nn.Parameter(
                torch.randn(
                    1,
                    1,
                    hidden
                )
            )

        else:

            self.cls_token = None

        self.pos_emb = nn.Parameter(
            torch.randn(
                1,
                num_tokens,
                hidden
            )
        )

        self.enc = nn.ModuleList([
            TransformerEncoder(
                feats=hidden,
                mlp_hidden=mlp_hidden,
                head=head,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.fc = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(
                hidden,
                num_classes
            )
        )

    def _to_words(self, x):

        B, C, H, W = x.shape

        p = self.patch_size

        x = (
            x.unfold(2, p, p)
             .unfold(3, p, p)
             .permute(
                 0, 2, 3, 4, 5, 1
             )
             .reshape(
                 B,
                 self.patch ** 2,
                 -1
             )
        )

        return x

    def forward(self, x):

        x = self._to_words(x)

        x = self.emb(x)

        if self.is_cls_token:

            cls = self.cls_token.repeat(
                x.shape[0],
                1,
                1
            )

            x = torch.cat(
                [cls, x],
                dim=1
            )

        x = x + self.pos_emb

        for enc in self.enc:

            x = enc(x)

        if self.is_cls_token:

            x = x[:, 0]

        else:

            x = x.mean(dim=1)

        return self.fc(x)


# ============================================================
# 4. LOAD FP32 CHECKPOINT
# ============================================================

fp32_model = ViT(
    img_size=32,
    patch=8,
    num_layers=2,
    hidden=96,
    mlp_hidden=96,
    head=4,
    is_cls_token=True,
    dropout=0.0,
    num_classes=10
).to(device)

fp32_checkpoint = torch.load(
    FP32_PATH,
    map_location="cpu",
    weights_only=False
)

fp32_model.load_state_dict(
    fp32_checkpoint["model"],
    strict=True
)

fp32_model.eval()


# ============================================================
# 5. LOAD INT8 CHECKPOINT
# ============================================================

int8_checkpoint = torch.load(
    INT8_PATH,
    map_location="cpu",
    weights_only=False
)

int8_weights = int8_checkpoint["model"]
weight_scales = int8_checkpoint["weight_scales"]

print("\nINT8 checkpoint loaded.")

print(
    "INT8 tensors:",
    len(int8_weights)
)

print(
    "Weight scales:",
    len(weight_scales)
)


# ============================================================
# 6. DEQUANTIZE INT8 WEIGHTS
# ============================================================

int8_model = ViT(
    img_size=32,
    patch=8,
    num_layers=2,
    hidden=96,
    mlp_hidden=96,
    head=4,
    is_cls_token=True,
    dropout=0.0,
    num_classes=10
).to(device)

int8_state = int8_model.state_dict()

loaded = 0

for name in int8_state.keys():

    q = int8_weights[name]

    # Floating-point parameters were quantized.
    if name in weight_scales:

        scale = weight_scales[name]

        reconstructed = (
            q.float() * scale
        )

        int8_state[name].copy_(
            reconstructed.to(device)
        )

    else:

        # Non-quantized tensor
        int8_state[name].copy_(
            q.to(device)
        )

    loaded += 1


int8_model.load_state_dict(
    int8_state,
    strict=True
)

int8_model.eval()

print(
    "Loaded tensors:",
    loaded
)


# ============================================================
# 7. EXACT CIFAR-10 TEST SET
# ============================================================

mean = [
    0.4914,
    0.4822,
    0.4465
]

std = [
    0.2470,
    0.2435,
    0.2616
]

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_dataset = datasets.CIFAR10(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 8. EVALUATION FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()


def evaluate(model):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item()
                * labels.size(0)
            )

            correct += (
                outputs.argmax(dim=1)
                == labels
            ).sum().item()

            total += labels.size(0)

    return (
        total_loss / total,
        correct / total
    )


# ============================================================
# 9. RUN FP32
# ============================================================

print("\nEvaluating FP32...")

fp32_loss, fp32_acc = evaluate(
    fp32_model
)


# ============================================================
# 10. RUN INT8
# ============================================================

print("Evaluating INT8...")

int8_loss, int8_acc = evaluate(
    int8_model
)


# ============================================================
# 11. LOGIT COMPARISON
# ============================================================

images, labels = next(
    iter(test_loader)
)

images = images.to(device)

with torch.no_grad():

    fp32_logits = fp32_model(
        images
    )

    int8_logits = int8_model(
        images
    )

logit_diff = (
    fp32_logits - int8_logits
).abs()

max_logit_diff = (
    logit_diff.max().item()
)

mean_logit_diff = (
    logit_diff.mean().item()
)


# ============================================================
# 12. FINAL METRICS
# ============================================================

num_parameters = sum(
    p.numel()
    for p in fp32_model.parameters()
)

fp32_mb = (
    num_parameters * 4
    / (1024 ** 2)
)

int8_mb = (
    num_parameters
    / (1024 ** 2)
)

compression = (
    fp32_mb / int8_mb
)

accuracy_drop = (
    fp32_acc - int8_acc
) * 100


# ============================================================
# 13. RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("C201 FP32 vs INT8")
print("=" * 70)

print(
    f"Parameters          : {num_parameters:,}"
)

print(
    f"FP32 accuracy       : {fp32_acc * 100:.2f}%"
)

print(
    f"INT8 accuracy       : {int8_acc * 100:.2f}%"
)

print(
    f"Accuracy drop       : {accuracy_drop:.2f} pp"
)

print(
    f"FP32 loss           : {fp32_loss:.6f}"
)

print(
    f"INT8 loss           : {int8_loss:.6f}"
)

print(
    f"Max logit difference: {max_logit_diff:.6f}"
)

print(
    f"Mean logit difference: {mean_logit_diff:.6f}"
)

print(
    f"FP32 weight memory  : {fp32_mb:.6f} MB"
)

print(
    f"INT8 weight memory  : {int8_mb:.6f} MB"
)

print(
    f"Compression         : {compression:.2f}x"
)

print("=" * 70)

Device: cuda

INT8 checkpoint loaded.
INT8 tensors: 40
Weight scales: 40
Loaded tensors: 40

Evaluating FP32...
Evaluating INT8...


C201 FP32 vs INT8
Parameters          : 124,714
FP32 accuracy       : 76.93%
INT8 accuracy       : 76.76%
Accuracy drop       : 0.17 pp
FP32 loss           : 0.725030
INT8 loss           : 0.725127
Max logit difference: 0.255602
Mean logit difference: 0.027019
FP32 weight memory  : 0.475746 MB
INT8 weight memory  : 0.118937 MB
Compression         : 4.00x
